In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm

cwd = Path.cwd()
ROOT = cwd if (cwd / "requirements.txt").exists() else cwd.parent
if not (ROOT / "requirements.txt").exists():
    raise FileNotFoundError("Project root not found")

PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "reports" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

actual = pd.read_csv(PROCESSED / "btcusdt_tr_vpin_with_u2_2020_q1.csv")
actual["available_at"] = pd.to_datetime(actual["available_at"], utc=True)
bk = actual.sort_values("bucket_id").reset_index(drop=True)
times = pd.DatetimeIndex(bk["available_at"], name="available_at")

train = pd.read_csv(PROCESSED / "forecast_train.csv")
test = pd.read_csv(PROCESSED / "forecast_test.csv")
for frame in (train, test):
    for column in ["timestamp", "target_end"]:
        frame[column] = pd.to_datetime(frame[column], utc=True)

signals = {}
for kind, column in [("tr", "oi"), ("u2", "u2")]:
    base = pd.Series(bk[column].to_numpy(dtype=float), index=times)

    for buckets in [50, 100]:                      # windows in bucket counts
        signals[f"{kind}_L{buckets}"] = base.rolling(buckets).mean()

    for hours in [6, 12, 24]:                      # windows in calendar hours
        s = base.rolling(f"{hours}h", min_periods=5).mean()
        s[s.index < times[0] + pd.Timedelta(hours=hours)] = np.nan
        signals[f"{kind}_cal{hours}h"] = s

# consistency with the baseline series
assert np.allclose(
    signals["tr_L50"].to_numpy(), bk["tr_vpin"].to_numpy(),
    equal_nan=True, rtol=1e-9, atol=1e-9,
)


def attach(frame, series, name):
    s = series.dropna()
    s = s[~s.index.duplicated(keep="last")]
    s = s.rename(name).reset_index().rename(columns={"available_at": f"t_{name}"})
    out = pd.merge_asof(
        frame.sort_values("timestamp"), s,
        left_on="timestamp", right_on=f"t_{name}", direction="backward",
    )
    matched = out[f"t_{name}"].notna()
    assert (out.loc[matched, f"t_{name}"] <= out.loc[matched, "timestamp"]).all()
    return out


panel_train, panel_test = train.copy(), test.copy()
for name, series in signals.items():
    panel_train = attach(panel_train, series, name)
    panel_test = attach(panel_test, series, name)

names = list(signals)
common_train = panel_train.dropna(subset=names).copy()
common_test = panel_test.dropna(subset=names).copy()
print("Common train / test rows:", len(common_train), len(common_test))

TARGET = "rv_next_1h_pct"
CONTROLS = ["rv_prev_1h_pct", "log_volume_prev_1h"]
y = common_test[TARGET].to_numpy(dtype=float)


def forecast(features):
    X_train = sm.add_constant(common_train[features].astype(float), has_constant="add")
    X_test = sm.add_constant(
        common_test[features].astype(float), has_constant="add"
    ).reindex(columns=X_train.columns)
    fit = sm.OLS(common_train[TARGET].astype(float), X_train, missing="raise").fit()
    return np.asarray(fit.predict(X_test))


mse_controls = float(np.mean((forecast(CONTROLS) - y) ** 2))
rows = []
for name in names:
    pred = forecast(CONTROLS + [name])
    rows.append({
        "signal": name,
        "n_train": len(common_train),
        "n_test": len(common_test),
        "mean_signal_train": common_train[name].mean(),
        "mean_signal_test": common_test[name].mean(),
        "MSE_gain_vs_controls_pct": 100 * (1 - np.mean((pred - y) ** 2) / mse_controls),
    })

table = pd.DataFrame(rows)
print(table.round(4).to_string(index=False))
table.to_csv(TABLES / "calendar_window_robustness.csv", index=False)

Common train / test rows: 1361 740
   signal  n_train  n_test  mean_signal_train  mean_signal_test  MSE_gain_vs_controls_pct
   tr_L50     1361     740             0.2408            0.2907                    0.6211
  tr_L100     1361     740             0.2450            0.2993                    2.7965
 tr_cal6h     1361     740             0.2213            0.2870                   -0.2939
tr_cal12h     1361     740             0.2314            0.2947                   -0.1112
tr_cal24h     1361     740             0.2392            0.2998                    1.1617
   u2_L50     1361     740             0.2763            0.3511                    1.7187
  u2_L100     1361     740             0.2804            0.3578                    4.1099
 u2_cal6h     1361     740             0.2567            0.3410                   -0.0867
u2_cal12h     1361     740             0.2657            0.3510                    0.4081
u2_cal24h     1361     740             0.2732            0.3576  

## Interpretation

Exploratory analysis, designed after viewing March results.

- TR-VPIN gain grows with the window length in calendar hours (−0.29%, −0.11%, 1.16% for
  6, 12, 24 hours) and with the number of buckets (0.62% for L = 50, 2.80% for L = 100).
- The 24-hour calendar window does not reproduce the L = 100 result, so the change of bucket
  frequency in March is not the only explanation.
- U2-VPIN gives a larger gain than TR-VPIN in all five window definitions.